# Train/Test Network Hardware Experiments (IBM Sherbrooke)

This notebook runs quantum IDS experiments on **IBM Sherbrooke fake backend** with realistic noise model.

**Key Features:**
- Simulates IBM Sherbrooke quantum hardware (127 qubits)
- Realistic noise model from actual IBM hardware
- Shot-based execution (1024 shots)
- Transpilation with connectivity constraints
- Direct comparison with simulator results

**Models:**
- Classical: LogReg, RandomForest, XGBoost/LightGBM
- Quantum (Hardware): QSVM, QNN variants, QKNN
- Fusion: Meta-ensemble models

**Configuration Options:**
1. `config_fast`: QSVM + QNN only (~2-3 hours)
2. `config_full`: All models including QKNN (~4-6 hours)

In [1]:
import os
import sys
import pandas as pd

# Add hardware pipeline to path
sys.path.append('/home/ritvik/Python/Copilot Files/Hardware/utils')
from utils.hardware_quantum_pipeline import HardwareConfig, run_dataset_experiment_hardware

## Configuration

Choose between two configurations:
- **Fast**: QSVM + QNN only (faster, covers main models)
- **Full**: All models including QKNN (comprehensive, takes longer)

In [2]:
# Option 1: QSVM + QNN only (faster execution)
config_fast = HardwareConfig(
    seed=42,
    num_features=13,  # Same as simulator for direct comparison
    shots=1024,       # Fixed at 1024
    backend_name="fake_sherbrooke",
    optimization_level=2,
    max_workers=max(1, (os.cpu_count() or 4) // 2),
    paper_fontsize=12,
    export_dpi=600,
    show_progress=True,
    # QSVM configuration (10 batches)
    qsvm_train_size=300,
    qsvm_test_size=100,
    # QNN configuration
    qnn_train_size=3000,
    qnn_test_size=1000,
    # Model selection
    run_qnn_models=True,
    run_qknn=False,  # Skip QKNN for faster execution
)

# Option 2: All models (complete comparison)
config_full = HardwareConfig(
    seed=42,
    num_features=13,
    shots=1024,
    backend_name="fake_sherbrooke",
    optimization_level=2,
    max_workers=max(1, (os.cpu_count() or 4) // 2),
    paper_fontsize=12,
    export_dpi=600,
    show_progress=True,
    qsvm_train_size=300,
    qsvm_test_size=100,
    qnn_train_size=3000,
    qnn_test_size=1000,
    run_qnn_models=True,
    run_qknn=True,  # Include QKNN
)

# Select which config to use
config = config_fast  # Change to config_full for complete comparison
config

HardwareConfig(seed=42, test_size=0.2, num_features=13, max_workers=8, target_fprs=(0.001, 0.01), cicids_sample_n=20000, cicids_max_files=None, qsvm_train_size=300, qsvm_test_size=100, qsvm_c_grid=(0.1, 1.0, 10.0, 20.0, 30.0), qnn_train_size=3000, qnn_test_size=1000, qnn_layers=5, qnn_epochs=35, qnn_batch_size=32, qnn_lr=0.01, vqc_train_size=2000, vqc_test_size=1000, vqc_layers=4, vqc_epochs=30, qknn_train_size=400, qknn_test_size=200, qknn_k=7, export_dpi=600, paper_fontsize=12, output_root='/home/ritvik/Python/Copilot Files/Hardware/results', plots_pdf_dir='/home/ritvik/Python/Plots PDF', images_dir='/home/ritvik/Python/Images', show_progress=True, backend_name='fake_sherbrooke', shots=1024, optimization_level=2, coupling_map_aware=True, noise_model_enabled=True, run_qnn_models=True, run_qknn=False)

## Run Hardware Experiment

This cell runs the complete experiment on IBM Sherbrooke fake backend.

**Expected runtime:**
- config_fast: ~2-3 hours
- config_full: ~4-6 hours

**What happens:**
1. Load and preprocess train_test_network.csv
2. Run classical baselines (LogReg, RF, XGBoost)
3. Run QSVM on hardware (10 batches × 300/100 samples)
4. Run QNN models on hardware (if enabled)
5. Run QKNN on hardware (if enabled)
6. Generate fusion meta-models
7. Export ROC/PR plots
8. Save results to Hardware/results/

In [3]:
results_df = run_dataset_experiment_hardware(
    dataset_name='train_test_network_hardware',
    config=config,
    train_test_csv_path='/home/ritvik/Python/Datasets/train_test_network.csv'
)
results_df


Starting Hardware Experiment: train_test_network_hardware
[    0.0s] Backend: fake_sherbrooke, Shots: 1024, Features: 13
[    0.0s] QNN models: enabled, QKNN: disabled
[    0.0s] Loading train_test_network dataset
[    3.1s] Data prepared: train=168834, test=42209, dim=13

Classical baselines (hardware comparison)
[    3.1s] Classical baseline: Logistic Regression
[    3.1s] LogReg: CV tuning started.
[   45.9s] LogReg: best params = {'C': 0.1}
[   46.3s] Classical baseline: Random Forest
[   46.3s] RandomForest: CV tuning started.
[  591.2s] RandomForest: best params = {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400}
[  591.8s] Classical baseline: XGBoost (fallback to LightGBM)
[  592.0s] XGBoost: CV tuning started.
[  602.7s] XGBoost: best params = {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 350}

Quantum kernel model (QSVM) - HARDWARE
[  606.5s] QSVM Hardware: processing 10 batches on fake_sherbrooke (1024 shots)


QSVM HW batches:   0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Results Summary

Display key metrics sorted by AUPRC (Area Under Precision-Recall Curve).

**Expected observations:**
- Hardware results should show 5-15% degradation vs. simulator due to noise
- Shot noise introduces variance in predictions
- Classical models remain unaffected (they don't use quantum hardware)

In [ ]:
summary_cols = [
    'Dataset', 'Model', 'Accuracy', 'F1', 'AUPRC', 'ROC_AUC',
    'TPR@0.1%FPR', 'TPR@1%FPR', 'Backend', 'Shots', 'Status'
]
available_cols = [c for c in summary_cols if c in results_df.columns]
results_df[available_cols].sort_values(by=['AUPRC'], ascending=False, na_position='last')

,Dataset,Model,Accuracy,F1,AUPRC,ROC_AUC,TPR@0.1%FPR,TPR@1%FPR,Status
2,train_test_network_hardware,XGBoost,0.999597,0.999736,1.000000,0.999999,0.999814,1.000000,NaN
1,train_test_network_hardware,RandomForest,0.999692,0.999798,1.000000,0.999999,0.999907,1.000000,NaN
0,train_test_network_hardware,LogReg,0.952404,0.969003,0.959241,0.955419,0.002887,0.005868,NaN
3,train_test_network_hardware,QSVM,NaN,NaN,NaN,NaN,NaN,NaN,SKIPPED: cannot import name 'FakeTorinoV2' fro...
4,train_test_network_hardware,QNN/VQC,NaN,NaN,NaN,NaN,NaN,NaN,SKIPPED: cannot import name 'FakeTorinoV2' fro...
5,train_test_network_hardware,MetaRF_Fusion,NaN,NaN,NaN,NaN,NaN,NaN,"SKIPPED: required outputs missing (QSVM, QNN_B..."


## Hardware vs. Simulator Comparison

Load simulator results and compare with hardware results.

In [ ]:
# Load simulator results for comparison
sim_results_path = '/home/ritvik/Python/Copilot Files/results/train_test_network_metrics_summary.csv'

if os.path.exists(sim_results_path):
    sim_df = pd.read_csv(sim_results_path)
    
    # Compare quantum models only
    quantum_models = ['QSVM', 'QNN_Base', 'QNN_Reupload', 'VQC_AltAnsatz', 'QKNN']
    
    comparison = []
    for model in quantum_models:
        sim_row = sim_df[sim_df['Model'] == model]
        hw_row = results_df[results_df['Model'] == model]
        
        if not sim_row.empty and not hw_row.empty:
            comparison.append({
                'Model': model,
                'Sim_Accuracy': sim_row['Accuracy'].values[0],
                'HW_Accuracy': hw_row['Accuracy'].values[0],
                'Accuracy_Delta': hw_row['Accuracy'].values[0] - sim_row['Accuracy'].values[0],
                'Sim_F1': sim_row['F1'].values[0],
                'HW_F1': hw_row['F1'].values[0],
                'F1_Delta': hw_row['F1'].values[0] - sim_row['F1'].values[0],
                'Sim_AUPRC': sim_row['AUPRC'].values[0],
                'HW_AUPRC': hw_row['AUPRC'].values[0],
                'AUPRC_Delta': hw_row['AUPRC'].values[0] - sim_row['AUPRC'].values[0],
            })
    
    comparison_df = pd.DataFrame(comparison)
    print("\n=== Simulator vs. Hardware Comparison ===")
    print("Negative delta = hardware degradation due to noise\n")
    comparison_df
else:
    print("Simulator results not found. Run 01_train_test_network_experiments.ipynb first.")


=== Simulator vs. Hardware Comparison ===
Negative delta = hardware degradation due to noise



## Next Steps

1. **Run CICIDS2017 hardware experiment**: Execute `02_hardware_cicids2017.ipynb`
2. **Analyze hardware characteristics**: Run `03_hardware_analysis.ipynb` for detailed IBM Sherbrooke analysis
3. **Review plots**: Check `Plots PDF/` and `Images/` folders for ROC/PR curves
4. **Conference paper**: Use results from hardware analysis notebook for publication